[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/09_sequential/09_proyecto_sasrec_cinematch.ipynb)

# Proyecto 09 · «Siguiente película» para CineMatch con SASRec en GPU

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | 3–4 h |
| **GPU** | L4 recomendada (A100 para los retos extra) |
| **Unidades Colab** | ≈ 2–4 (L4) |
| **Prerrequisitos** | Lección 09 |

## 🏢 Contexto de negocio
Eres ML engineer en **CineMatch**. La fila de la home *«Porque acabas de ver…»* usa hoy un item-kNN estático (módulo 04) y el equipo de producto observa que
**ignora lo que el usuario está viendo esta semana**. Tu misión: construir el modelo de **siguiente película** que alimentará esa fila (y, más adelante, la etapa de *retrieval* del capstone).

Producto te pide una comparación honesta entre un GRU4Rec (barato) y un SASRec (más caro de servir) para decidir qué desplegar.

## 📦 Dataset
**MovieLens-1M** (GroupLens): 6.040 usuarios, ~3.700 películas, 1 M de ratings con *timestamp*. Lo tratamos como feedback implícito ordenado por tiempo.
Protocolo: *leave-one-out* (último ítem = test, penúltimo = validación), *full ranking*, enmascarando lo ya visto.

## ✅ Entregables y rúbrica
| Criterio | Objetivo |
|---|---|
| Popularidad y Markov implementados y evaluados | obligatorio |
| GRU4Rec entrenado (CE) | NDCG@10 test > Markov |
| SASRec entrenado con **CE completa** | **NDCG@10 ≥ 0,15** y HR@10 ≥ 0,27 en test (orientativo, ML-1M real) |
| Comparación BCE(1 neg) vs gBCE vs CE en SASRec | tabla + gráfico + 3 frases de interpretación |
| Intervalo de confianza *bootstrap* del ΔNDCG@10 SASRec − GRU4Rec | IC 95 % reportado |
| Recomendación final a producto (coste vs calidad) | ≤ 10 líneas |

In [ ]:
!pip install -q seaborn

In [ ]:
import math, time, random, collections, subprocess
import numpy as np, pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda" and torch.cuda.is_bf16_supported()   # bf16 en L4/A100 (no en T4)
FAST_DEV_RUN = False          # True = iterar rápido (pocas épocas, subconjunto); False = experimento completo en GPU
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("device:", device, "| bf16:", USE_AMP, "| torch", torch.__version__)

## 🧰 Utilidades mínimas (vienen de los módulos 01 y 02)

Para que este notebook sea **autocontenido en Colab**, copiamos aquí una versión reducida de las utilidades que construimos en
`01_data` (carga de MovieLens, secuencias, *splits*) y `02_evaluation` (HR@K y NDCG@K con *full ranking*).

- Si la descarga falla (sin red, *firewall*…), se generan **datos sintéticos con estructura secuencial** para que el notebook nunca se bloquee. Los números con datos sintéticos **no** son comparables con los papers.
- Convención de IDs: los ítems se re-indexan a `1..n_items` y **el 0 es el *padding***.

In [ ]:
# ==== Utilidades (versión mínima de 01_data) ====
import os, io, zipfile, urllib.request

ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

def make_synthetic_ml(n_users=1500, n_items=700, n_genres=12, seed=42):
    '''Datos de juguete con señal secuencial: "siguiente capítulo", deriva de género y popularidad.'''
    rng = np.random.default_rng(seed)
    genre = rng.integers(0, n_genres, n_items)
    by_genre = [np.where(genre == g)[0] for g in range(n_genres)]
    pop = np.minimum(rng.zipf(1.4, n_items), 60).astype(float); pop /= pop.sum()
    rows = []
    for u in range(1, n_users + 1):
        g = int(rng.integers(n_genres)); it = int(rng.choice(by_genre[g])); t = 978_300_000 + int(rng.integers(0, 10**7))
        for _ in range(int(rng.integers(20, 160))):
            rows.append((u, it + 1, int(rng.integers(3, 6)), t))
            t += int(rng.integers(30, 3 * 86400)); r = rng.random()
            if r < 0.45:   it = (it + int(rng.integers(1, 4))) % n_items          # secuela / siguiente capítulo
            elif r < 0.8:  it = int(rng.choice(by_genre[g]))                       # mismo género
            else:          g = int(rng.integers(n_genres)); it = int(rng.choice(n_items, p=pop))
    ratings = pd.DataFrame(rows, columns=["user", "item", "rating", "ts"]).drop_duplicates(["user", "item"])
    names = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance", "Thriller",
             "Animation", "Documentary", "Crime", "Musical", "War"]
    movies = pd.DataFrame({"item": np.arange(1, n_items + 1),
                           "title": [f"Película sintética {i} (2000)" for i in range(1, n_items + 1)],
                           "genres": [names[g % len(names)] for g in genre]})
    return ratings, movies

def load_movielens_1m(data_dir="data"):
    '''Devuelve (ratings[user,item,rating,ts], movies[item,title,genres]).'''
    os.makedirs(data_dir, exist_ok=True)
    zpath = os.path.join(data_dir, "ml-1m.zip")
    try:
        if not os.path.exists(zpath):
            urllib.request.urlretrieve(ML1M_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            ratings = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python",
                                  names=["user", "item", "rating", "ts"])
            movies = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python",
                                 names=["item", "title", "genres"], encoding="latin-1")
        return ratings, movies
    except Exception as e:
        if os.path.exists(zpath):
            os.remove(zpath)
        print(f"⚠️ No se pudo descargar MovieLens-1M ({type(e).__name__}). Uso datos SINTÉTICOS.")
        return make_synthetic_ml()

In [ ]:
# ==== Secuencias y split leave-one-out (01_data) ====
def build_sequences(ratings, min_len=5):
    '''Ordena por tiempo y devuelve (seqs: list[list[int]], item2idx, idx2item). 0 = padding.'''
    df = ratings.sort_values(["user", "ts"], kind="mergesort")      # estable: respeta el orden del fichero en empates
    cnt = df.groupby("user")["item"].transform("size")
    df = df[cnt >= min_len]
    items = np.sort(df["item"].unique())
    item2idx = {it: i + 1 for i, it in enumerate(items)}
    idx2item = np.concatenate([[0], items])
    df = df.assign(iidx=df["item"].map(item2idx))
    seqs = df.groupby("user", sort=True)["iidx"].apply(list).tolist()
    return seqs, item2idx, idx2item

def leave_one_out(seqs):
    '''train = s[:-2]; valid: (s[:-2] -> s[-2]); test: (s[:-1] -> s[-1]).'''
    train = [s[:-2] for s in seqs]
    valid = ([s[:-2] for s in seqs], [s[-2] for s in seqs])
    test = ([s[:-1] for s in seqs], [s[-1] for s in seqs])
    return train, valid, test

def pad_left(seqs, maxlen):
    '''Trunca a los últimos `maxlen` y rellena con 0 por la izquierda -> LongTensor [B, maxlen].'''
    out = np.zeros((len(seqs), maxlen), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[-maxlen:]
        if len(s):
            out[i, -len(s):] = s
    return torch.from_numpy(out)

In [ ]:
# ==== Métricas HR@K / NDCG@K con full ranking (02_evaluation) ====
@torch.no_grad()
def evaluate_next_item(score_fn, histories, targets, ks=(10,), batch_size=512, mask_history=True):
    '''score_fn(list[list[int]]) -> Tensor [B, n_items+1] con puntuaciones de TODOS los ítems.
    Ranking completo (sin negativos muestreados, ver Krichene & Rendle 2020). Una sola relevante => HR = Recall.'''
    ranks = []
    for b in range(0, len(histories), batch_size):
        H, T = histories[b:b + batch_size], torch.as_tensor(targets[b:b + batch_size])
        s = score_fn(H).float().cpu()
        tgt = s.gather(1, T[:, None]).clone()
        s[:, 0] = -float("inf")                                # padding
        if mask_history:                                       # no recomendar lo ya visto
            for i, h in enumerate(H):
                s[i, h[0] if isinstance(h, tuple) else h] = -float("inf")   # h puede ser (items, timestamps)
        s.scatter_(1, T[:, None], tgt)                         # el objetivo nunca se enmascara
        ranks.append((s > tgt).sum(1))                         # posición 0-based (empates optimistas)
    ranks = torch.cat(ranks).numpy()
    out = {}
    for k in ks:
        hit = ranks < k
        out[f"HR@{k}"] = float(hit.mean())
        out[f"NDCG@{k}"] = float((hit / np.log2(ranks + 2)).mean())
    return out

In [ ]:
ratings, movies = load_movielens_1m()
seqs, item2idx, idx2item = build_sequences(ratings, min_len=5)
n_items = len(item2idx)
train_seqs, valid, test = leave_one_out(seqs)
title_of = dict(zip(movies["item"], movies["title"]))
genre_of = dict(zip(movies["item"], movies["genres"].str.split("|").str[0]))
idx_title = lambda i: title_of.get(int(idx2item[i]), str(i))
lens = np.array([len(s) for s in seqs])
print(f"usuarios={len(seqs):,}  ítems={n_items:,}  interacciones={lens.sum():,}  "
      f"long. mediana={np.median(lens):.0f}  máx={lens.max()}")

## Paso 1 · Baselines (popularidad y Markov)
Implementa dos `score_fn` compatibles con `evaluate_next_item`: reciben una lista de historiales y devuelven un tensor `[B, n_items+1]`.

In [ ]:
def popularity_score_fn(train_seqs, n_items):
    # TODO: cuenta apariciones de cada ítem en train y devuelve una función H -> tensor [len(H), n_items+1]
    # Pista: np.bincount(np.concatenate(train_seqs), minlength=n_items+1)
    raise NotImplementedError

def markov_score_fn(train_seqs, n_items, eps=1e-3):
    # TODO: matriz dispersa de transiciones i -> j (consecutivas) y puntuación = fila del último ítem + eps * popularidad
    raise NotImplementedError

print(evaluate_next_item(popularity_score_fn(train_seqs, n_items), *valid))
print(evaluate_next_item(markov_score_fn(train_seqs, n_items), *valid))

## Paso 2 · Modelos
Implementa `GRU4Rec` y `SASRec` con la interfaz `encode(seq) -> [B, L, d]` y atributos `n_items`, `item_emb` (tamaño `n_items + 2`, 0 = padding).
Pistas: máscara causal con `torch.triu(..., 1)`, no atender a padding, permitir la diagonal; LayerNorm final.

In [ ]:
class GRU4Rec(nn.Module):
    def __init__(self, n_items, d=64, hidden=128, dropout=0.2):
        super().__init__()
        # TODO
        raise NotImplementedError

    def encode(self, seq):
        raise NotImplementedError


class SASRec(nn.Module):
    def __init__(self, n_items, d=64, n_layers=2, n_heads=2, maxlen=200, dropout=0.2):
        super().__init__()
        # TODO: item_emb, pos_emb, bloques Transformer (puedes usar nn.TransformerEncoderLayer con norm_first=True
        #       y una máscara float/bool combinada de forma [B*heads, L, L]) y LayerNorm final
        raise NotImplementedError

    def encode(self, seq):
        raise NotImplementedError

## Paso 3 · Pérdidas y entrenamiento
Implementa `seq_loss` con `"ce"`, `"bce"` y `"gbce"` (β = α(t(1−1/α)+1/α), α = k/(N−1)) y un bucle de entrenamiento que evalúe en validación cada pocas épocas.

In [ ]:
def seq_loss(model, h, tgt, loss="ce", n_neg=1, gbce_t=0.75):
    # TODO
    raise NotImplementedError

def train(model, loss="ce", epochs=100, n_neg=1, gbce_t=0.75, maxlen=200, batch_size=128, lr=1e-3):
    # TODO: devuelve historial con NDCG@10 de validación
    raise NotImplementedError

## Paso 4 · Experimentos y bootstrap
1. Entrena GRU4Rec (CE), SASRec (BCE, 1 neg), gSASRec (gBCE, 256 neg, t = 0,75) y SASRec+ (CE).
2. Evalúa en test. 3. Calcula un IC 95 % *bootstrap* (1.000 remuestreos de usuarios) de ΔNDCG@10 entre SASRec+ y GRU4Rec.
Pista: modifica `evaluate_next_item` (o copia su lógica) para devolver el NDCG **por usuario**.

In [ ]:
def per_user_ndcg(score_fn, histories, targets, k=10):
    # TODO: vector de NDCG@k por usuario
    raise NotImplementedError

def bootstrap_ci(a, b, n_boot=1000, seed=42):
    # TODO: IC 95 % de mean(a - b) remuestreando usuarios con reemplazo
    raise NotImplementedError

---
# ⛔ SPOILER — Solución de referencia
Intenta resolverlo primero. La solución reutiliza el código de la lección (versión compacta).

In [ ]:
# --- Solución: baselines ---
def popularity_score_fn(train_seqs, n_items):
    pop = np.bincount(np.concatenate(train_seqs), minlength=n_items + 1).astype(np.float32)
    return lambda H: torch.from_numpy(np.tile(pop, (len(H), 1)))

def markov_score_fn(train_seqs, n_items, eps=1e-3):
    src = np.concatenate([s[:-1] for s in train_seqs if len(s) > 1]); dst = np.concatenate([s[1:] for s in train_seqs if len(s) > 1])
    M = sp.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(n_items + 1, n_items + 1))
    pop = np.bincount(np.concatenate(train_seqs), minlength=n_items + 1).astype(np.float32); pop /= pop.sum()
    return lambda H: torch.from_numpy(M[[h[-1] for h in H]].toarray() + eps * pop)

base_res = {"Popularidad": evaluate_next_item(popularity_score_fn(train_seqs, n_items), *test, ks=(10,)),
            "Markov": evaluate_next_item(markov_score_fn(train_seqs, n_items), *test, ks=(10,))}
print(base_res)

In [ ]:
class GRU4Rec(nn.Module):
    '''GRU4Rec simplificado: embedding -> GRU -> proyección; salida con pesos compartidos (weight tying).'''
    def __init__(self, n_items, d=64, hidden=128, n_layers=1, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.item_emb = nn.Embedding(n_items + 2, d, padding_idx=0)
        nn.init.normal_(self.item_emb.weight, std=0.02)
        self.gru = nn.GRU(d, hidden, n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.proj = nn.Linear(hidden, d)
        self.drop = nn.Dropout(dropout)

    def encode(self, seq):                     # seq [B, L] -> [B, L, d]
        h, _ = self.gru(self.drop(self.item_emb(seq)))
        return self.proj(self.drop(h))


class SelfAttention(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        assert d % n_heads == 0
        self.h, self.dk = n_heads, d // n_heads
        self.qkv, self.out, self.drop = nn.Linear(d, 3 * d), nn.Linear(d, d), nn.Dropout(dropout)
        self.keep_attn, self.last_attn = False, None

    def forward(self, x, block):               # block: bool [B,1,L,L]; True = prohibido atender
        B, L, D = x.shape
        q, k, v = self.qkv(x).view(B, L, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        att = att.masked_fill(block, float("-inf")).softmax(-1)
        if self.keep_attn:
            self.last_attn = att.detach().float().cpu()
        y = self.drop(att) @ v
        return self.out(y.transpose(1, 2).reshape(B, L, D))


class Block(nn.Module):
    '''Bloque Transformer pre-LN: x + Attn(LN(x)); x + FFN(LN(x)).'''
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = SelfAttention(d, n_heads, dropout)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, block):
        x = x + self.drop(self.attn(self.ln1(x), block))
        return x + self.drop(self.ffn(self.ln2(x)))

In [ ]:
def attention_block_mask(seq, causal=True):
    '''True = prohibido. Bloquea padding (claves) y, si causal, el futuro. La diagonal siempre se permite
    para que ninguna fila quede entera a -inf (evita NaN en las posiciones de padding).'''
    B, L = seq.shape
    blk = (seq == 0)[:, None, None, :].expand(B, 1, L, L).clone()
    if causal:
        blk |= torch.triu(torch.ones(L, L, dtype=torch.bool, device=seq.device), 1)
    return blk & ~torch.eye(L, dtype=torch.bool, device=seq.device)


class SASRec(nn.Module):
    '''SASRec (Kang & McAuley 2018). Con causal=False y un token [MASK] se convierte en BERT4Rec.'''
    def __init__(self, n_items, d=64, n_layers=2, n_heads=2, maxlen=200, dropout=0.2, causal=True):
        super().__init__()
        self.n_items, self.d, self.causal = n_items, d, causal
        self.item_emb = nn.Embedding(n_items + 2, d, padding_idx=0)   # n_items+1 = [MASK]
        self.pos_emb = nn.Embedding(maxlen, d)
        nn.init.normal_(self.item_emb.weight, std=0.02); nn.init.normal_(self.pos_emb.weight, std=0.02)
        self.blocks = nn.ModuleList([Block(d, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f, self.drop = nn.LayerNorm(d), nn.Dropout(dropout)

    def encode(self, seq):                     # seq [B, L] (padding a la izquierda) -> [B, L, d]
        L = seq.size(1)
        x = self.item_emb(seq) + self.pos_emb(torch.arange(L, device=seq.device))
        x = self.drop(x) * (seq > 0).unsqueeze(-1)
        blk = attention_block_mask(seq, self.causal)
        for b in self.blocks:
            x = b(x, blk)
        return self.ln_f(x)

def n_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
def seq_loss(model, h, tgt, loss="ce", n_neg=1, gbce_t=0.75):
    '''h [B,L,d] estados ocultos; tgt [B,L] ítem siguiente (0 = ignorar).
    loss ∈ {"ce" (full softmax), "sampled_ce", "bce", "gbce"}.'''
    W, n, valid = model.item_emb.weight, model.n_items, tgt > 0
    if loss == "ce":                                                  # softmax sobre TODO el catálogo
        logits = h[valid] @ W[1:n + 1].T
        return F.cross_entropy(logits, tgt[valid] - 1)
    neg = torch.randint(1, n + 1, (h.size(0), n_neg), device=h.device)  # negativos uniformes por secuencia
    pos_l = (h * W[tgt]).sum(-1)[valid]                               # [N]
    neg_l = torch.einsum("bld,bkd->blk", h, W[neg])[valid]            # [N, k]
    if loss == "sampled_ce":                                          # softmax sobre {positivo} ∪ negativos
        logits = torch.cat([pos_l[:, None], neg_l], 1)
        return F.cross_entropy(logits, torch.zeros(len(pos_l), dtype=torch.long, device=h.device))
    alpha = n_neg / (n - 1)                                           # tasa de muestreo de negativos
    beta = 1.0 if loss == "bce" else alpha * (gbce_t * (1 - 1 / alpha) + 1 / alpha)
    return (-beta * F.logsigmoid(pos_l) - F.logsigmoid(-neg_l).sum(-1)).mean()

In [ ]:
def make_bert_batch(batch, maxlen, mask_prob, mask_token):
    '''Cloze: enmascara ~mask_prob de las posiciones; etiqueta = ítem original (0 en las no enmascaradas).'''
    seq = pad_left(batch, maxlen)
    m = (torch.rand(seq.shape) < mask_prob) & (seq > 0)
    m[:, -1] |= torch.rand(seq.size(0)) < 0.25        # a veces enmascara el último: imita la inferencia
    return seq.masked_fill(m, mask_token), seq * m

def make_score_fn(model, maxlen, bert=False):
    @torch.no_grad()
    def f(H):
        model.eval()
        H = [h + [model.n_items + 1] for h in H] if bert else H    # BERT4Rec: [MASK] al final
        h = model.encode(pad_left(H, maxlen).to(device))[:, -1]
        return h.float() @ model.item_emb.weight[: model.n_items + 1].float().T
    return f

def train_seq_model(model, train_seqs, val, loss="ce", epochs=50, maxlen=200, batch_size=128, lr=1e-3,
                    n_neg=1, gbce_t=0.75, bert=False, mask_prob=0.2, eval_every=5, n_val=2000, name="modelo"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    users = [i for i, s in enumerate(train_seqs) if len(s) >= 2]
    vi = np.random.default_rng(0).choice(len(val[0]), min(n_val, len(val[0])), replace=False)
    val_sub = ([val[0][i] for i in vi], [val[1][i] for i in vi])
    score_fn, hist, t0 = make_score_fn(model, maxlen, bert), collections.defaultdict(list), time.time()
    for ep in range(1, epochs + 1):
        model.train(); perm, tot = np.random.permutation(users), 0.0
        for b in range(0, len(perm), batch_size):
            batch = [train_seqs[i] for i in perm[b:b + batch_size]]
            if bert:
                inp, tgt = make_bert_batch(batch, maxlen, mask_prob, model.n_items + 1)
            else:
                inp, tgt = pad_left([s[:-1] for s in batch], maxlen), pad_left([s[1:] for s in batch], maxlen)
            inp, tgt = inp.to(device), tgt.to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_AMP):
                h = model.encode(inp)
            l = seq_loss(model, h.float(), tgt, "ce" if bert else loss, n_neg, gbce_t)
            opt.zero_grad(set_to_none=True); l.backward(); opt.step()
            tot += l.item() * len(batch)
        if ep % eval_every == 0 or ep == epochs:
            m = evaluate_next_item(score_fn, *val_sub, ks=(10,))
            hist["epoch"].append(ep); hist["loss"].append(tot / len(perm)); hist["val_NDCG@10"].append(m["NDCG@10"])
            print(f"[{name}] ep {ep:3d}  loss={tot/len(perm):.4f}  val NDCG@10={m['NDCG@10']:.4f}  "
                  f"({time.time()-t0:.0f}s)")
    hist["train_time_s"] = time.time() - t0
    return dict(hist)

In [ ]:
MAXLEN = 200
CFG = dict(epochs=3 if FAST_DEV_RUN else 100, maxlen=MAXLEN, batch_size=128, lr=1e-3, eval_every=1 if FAST_DEV_RUN else 10)
ARCH = dict(d=64, n_layers=2, n_heads=2, maxlen=MAXLEN, dropout=0.2)
N_NEG = min(256, n_items - 1)
exps = {"GRU4Rec (CE)": (lambda: GRU4Rec(n_items), dict(loss="ce")),
        "SASRec (BCE, 1 neg)": (lambda: SASRec(n_items, **ARCH), dict(loss="bce", n_neg=1)),
        "gSASRec (gBCE)": (lambda: SASRec(n_items, **ARCH), dict(loss="gbce", n_neg=N_NEG, gbce_t=0.75)),
        "SASRec+ (CE)": (lambda: SASRec(n_items, **ARCH), dict(loss="ce"))}
trained, hists, final = {}, {}, {}
for name, (ctor, kw) in exps.items():
    torch.manual_seed(seed); np.random.seed(seed)
    trained[name] = ctor()
    hists[name] = train_seq_model(trained[name], train_seqs, valid, name=name, **CFG, **kw)
    final[name] = evaluate_next_item(make_score_fn(trained[name], MAXLEN), *test, ks=(10,))
    final[name]["min"] = hists[name]["train_time_s"] / 60
final.update(base_res)
display(pd.DataFrame(final).T.sort_values("NDCG@10").round(4))

In [ ]:
@torch.no_grad()
def per_user_ndcg(score_fn, histories, targets, k=10, batch_size=512):
    out = []
    for b in range(0, len(histories), batch_size):
        H, T = histories[b:b + batch_size], torch.as_tensor(targets[b:b + batch_size])
        s = score_fn(H).float().cpu(); tgt = s.gather(1, T[:, None]).clone()
        s[:, 0] = -float("inf")
        for i, h in enumerate(H):
            s[i, h] = -float("inf")
        s.scatter_(1, T[:, None], tgt)
        r = (s > tgt).sum(1).numpy()
        out.append((r < k) / np.log2(r + 2))
    return np.concatenate(out)

def bootstrap_ci(a, b, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed); d = a - b
    boots = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
    return d.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

nd_sas = per_user_ndcg(make_score_fn(trained["SASRec+ (CE)"], MAXLEN), *test)
nd_gru = per_user_ndcg(make_score_fn(trained["GRU4Rec (CE)"], MAXLEN), *test)
delta, lo, hi = bootstrap_ci(nd_sas, nd_gru)
print(f"ΔNDCG@10 (SASRec+ − GRU4Rec) = {delta:.4f}   IC95% [{lo:.4f}, {hi:.4f}]")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, h in hists.items():
    ax[0].plot(h["epoch"], h["val_NDCG@10"], marker="o", ms=3, label=name)
ax[0].set(title="Validación NDCG@10", xlabel="época"); ax[0].legend(fontsize=8)
fr = pd.DataFrame(final).T.sort_values("NDCG@10")
fr["NDCG@10"].plot.barh(ax=ax[1], color="#6366f1"); ax[1].set(title="Test NDCG@10")
plt.tight_layout(); plt.show()

**Interpretación esperada (con ML-1M real):** SASRec+ (CE) > gSASRec ≳ GRU4Rec > SASRec-BCE(1 neg) > Markov > Popularidad.
La brecha BCE(1) → gBCE ilustra la sobreconfianza; la brecha gBCE → CE, el valor de ver todo el catálogo como negativo.
Si el IC del Δ no incluye 0, la mejora de SASRec+ sobre GRU4Rec es estadísticamente sólida para este *split*.

**Recomendación a producto (ejemplo):** desplegar SASRec+ como generador de candidatos de la fila «Porque acabas de ver…»: el coste de serving
(un *forward* de 2 capas sobre ≤200 ítems, ~ms en GPU, o precálculo por lotes cada hora) es asumible y la ganancia de NDCG es significativa.

## 🚀 Retos extra (nivel experto)
1. **Split temporal global**: re-evalúa todos los modelos cortando por fecha (módulo 01). ¿Se mantiene el ranking?
2. **Sampled softmax con logQ** con negativos muestreados por popularidad (módulo 08) y compáralo con gBCE.
3. **Features de contenido**: suma al embedding del ítem un embedding de géneros (como hace RecTools con `CatFeaturesItemNet`) y mide el efecto en ítems poco populares.
4. **Escala**: repite con MovieLens-20M o -32M en A100 (`d=128`, 4 capas, `maxlen=200`). ¿Crece la ventaja de SASRec?
5. **Exporta el encoder** a ONNX/TorchScript y mide la latencia p99 de inferencia para un usuario (preparación del módulo 16).

## 🤔 Reflexión (producción / MLOps)
- ¿Calcularías el embedding de usuario en cada petición (*stateless*) o en batch (como PinnerFormer)? ¿Qué frescura necesita esta fila?
- ¿Cómo detectarías en producción que el modelo se ha degradado (módulo 17)? Propón 2 métricas online y 1 offline.
- Los ítems nuevos no tienen embedding: ¿cómo los introducirías en el catálogo sin reentrenar (pista: módulo 03 y semantic IDs del módulo 11)?